In [1]:
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
import json

# Load all results from the new JSONL (line-based) file
results = []
with open("../results_llava-hf/llava-1.5-7b-hf/attention_results_all_levels.jsonl", "r") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))
print(f"Number of inferences: {len(results)}")

Number of inferences: 5368


In [ ]:
# Collect all grids for each (relation_type, src, tgt) pair
relation_pair_to_grids = {}

# Specify the group pairs you want to plot entropy for
selected_group_pairs = [
    # ("last", "all_visual"),
    ("relation", "all_visual"),
    # Add more (src, tgt) pairs as needed
]


for result in results:
    relation_type = result["relation_type"]  # Extract relation_type from each result
    attn_metrics = result["attention_metrics"]
    for pair in attn_metrics["group_pairs"]:
        src = pair["source_group"]
        tgt = pair["target_group"]
        if (src, tgt) not in selected_group_pairs:
            continue
        per_layer = pair["per_layer"]
        attn_grid = np.array([layer["per_head_fraction"] for layer in per_layer])  # (layers, heads)
        key = (relation_type, src, tgt)  # Include relation_type in the key
        relation_pair_to_grids.setdefault(key, []).append(attn_grid)

# Compute and plot the mean grid for each (relation_type, src, tgt) combination
# First, compute overall mean across all relations for each (src, tgt)
overall_means_frac = {}
for (relation, src, tgt), grids in relation_pair_to_grids.items():
    if (src, tgt) not in overall_means_frac:
        overall_means_frac[(src, tgt)] = []
    overall_means_frac[(src, tgt)].extend(grids)

for key, all_grids in overall_means_frac.items():
    overall_means_frac[key] = np.mean(all_grids, axis=0)

THRESHOLD = 0.5  # Set your desired threshold here
for (relation, src, tgt), grids in relation_pair_to_grids.items():
    mean_grid = np.mean(grids, axis=0)  # mean over all QAs for this relation and pair
    overall_mean_grid = overall_means_frac[(src, tgt)]
    diff_grid = mean_grid - overall_mean_grid  # Difference from overall mean
    
    # Swap axes: heads on y-axis, layers on x-axis
    mean_grid_swapped = mean_grid.T
    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(mean_grid_swapped, cmap="viridis", annot=False)
    # Mark cells above threshold with a hollow yellow circle
    for y in range(mean_grid_swapped.shape[0]):  # heads
        for x in range(mean_grid_swapped.shape[1]):  # layers
            if mean_grid_swapped[y, x] > THRESHOLD:
                circle = plt.Circle((x + 0.5, y + 0.5), 0.25, color="yellow", fill=False, linewidth=2)
                ax.add_patch(circle)
    plt.title(f"Mean Attention Fraction: {relation} | {src} → {tgt} (averaged over {len(grids)} QAs)\nYellow circle: > {THRESHOLD}")
    plt.xlabel("Layer")
    plt.ylabel("Head")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Plot Overall Mean for reference grid
    overall_mean_grid_swapped = overall_mean_grid.T
    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(overall_mean_grid_swapped, cmap="viridis", annot=False)  # Diverging colormap centered at 0
    plt.title(f"Overall Mean Attention | {src} → {tgt}")
    plt.xlabel("Layer")
    plt.ylabel("Head")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Plot difference grid
    diff_grid_swapped = diff_grid.T
    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(diff_grid_swapped, cmap="RdBu_r", annot=False, center=0)  # Diverging colormap centered at 0
    plt.title(f"Difference from Overall Mean Attention: {relation} | {src} → {tgt}")
    plt.xlabel("Layer")
    plt.ylabel("Head")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# How focussed are the "looking at" behaviours of the relational-phrase and the last tokens? An Attention Analysis

In [ ]:
# Collect all grids for each (src, tgt) pair (entropy version, filtered group pairs)

# Specify the group pairs you want to plot entropy for
selected_group_pairs = [
    # ("last", "all_visual"),
    ("relation", "all_visual"),
    # Add more (src, tgt) pairs as needed
]

relation_pair_to_entropy_grids = {}

for result in results:
    attn_metrics = result["attention_metrics"]
    relation_type = result["relation_type"]
    for pair in attn_metrics["group_pairs"]:
        src = pair["source_group"]
        tgt = pair["target_group"]
        if (src, tgt) not in selected_group_pairs:
            continue
        per_layer = pair["per_layer"]
        # Use entropy instead of fraction
        entropy_grid = np.array([layer["per_head_entropy"] for layer in per_layer])  # (layers, heads)
        print(f"Entropy grid shape: {entropy_grid.shape}, min: {entropy_grid.min():.4f}, max: {entropy_grid.max():.4f}")
        key = (relation_type, src, tgt)  # Include relation_type in the key
        relation_pair_to_entropy_grids.setdefault(key, []).append(entropy_grid)

# Compute and plot the mean entropy grid for each selected pair
# First, compute overall mean across all relations for each (src, tgt)
overall_means = {}
for (relation, src, tgt), grids in relation_pair_to_entropy_grids.items():
    if (src, tgt) not in overall_means:
        overall_means[(src, tgt)] = []
    overall_means[(src, tgt)].extend(grids)

for key, all_grids in overall_means.items():
    overall_means[key] = np.mean(all_grids, axis=0)

for (relation, src, tgt), grids in relation_pair_to_entropy_grids.items():
    mean_grid = np.mean(grids, axis=0)  # mean over all QAs
    overall_mean_grid = overall_means[(src, tgt)]
    diff_grid = mean_grid - overall_mean_grid  # Difference from overall mean
    
    # Plot mean grid
    mean_grid_swapped = mean_grid.T
    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(mean_grid_swapped, cmap="magma", annot=False)
    plt.title(f"Mean Attention Entropy: {relation} | {src} → {tgt} (averaged over {len(grids)} QAs)")
    plt.xlabel("Layer")
    plt.ylabel("Head")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Plot difference grid
    diff_grid_swapped = diff_grid.T
    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(diff_grid_swapped, cmap="RdBu_r", annot=False, center=0)  # Diverging colormap centered at 0
    plt.title(f"Difference from Overall Mean Entropy: {relation} | {src} → {tgt}")
    plt.xlabel("Layer")
    plt.ylabel("Head")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

Here, I plot violin curves to see distribution of entropy for a specific head in a layer, by relation type. This will be useful in seeing if this head behaves differently for different relation types. Candidate heads are taken from those that have different attention fraction to the mean. 

candidates = [(9, 8), (9, 9), (10, 10), (11, 17), (13, 19), (14, 24), (13, 23), (31, 22), (14, 2)] 

In [ ]:
# Specify the layers to plot
layers = range(0, 32)  # Change this to the desired layers

selected_group_pairs = [("relation", "all_visual")]

for selected_layer in layers:
    # Collect fractions for each head in the selected layer per relation type
    fractions_per_head_relation = {}

    for result in results:
        relation_type = result["relation_type"]
        attn_metrics = result["attention_metrics"]
        for pair in attn_metrics["group_pairs"]:
            src = pair["source_group"]
            tgt = pair["target_group"]
            if (src, tgt) not in selected_group_pairs:
                continue
            per_layer = pair["per_layer"]
            layer_data = per_layer[selected_layer]
            per_head_fraction = layer_data["per_head_fraction"]
            for head_idx, frac in enumerate(per_head_fraction):
                if head_idx not in fractions_per_head_relation:
                    fractions_per_head_relation[head_idx] = {}
                if relation_type not in fractions_per_head_relation[head_idx]:
                    fractions_per_head_relation[head_idx][relation_type] = []
                fractions_per_head_relation[head_idx][relation_type].append(frac)

    # Now plot violin plots for each head in the selected layer, with violins for each relation type
    num_heads = 32
    fig, axes = plt.subplots(4, 8, figsize=(20, 10))  # 4 rows, 8 columns for 32 heads
    axes = axes.flatten()
    
    for head in range(num_heads):
        ax = axes[head]

        # ---- FIX Y AXIS RANGE ----
        ax.set_ylim(0, 1)
        row = head // 8
        col = head % 8

        if head in fractions_per_head_relation:
            relation_fractions = fractions_per_head_relation[head]
            data = list(relation_fractions.values())
            labels = list(relation_fractions.keys())
            sns.violinplot(data=data, ax=ax, inner="quartile")

            # ---- X AXIS: only bottom row ----
            if row == 3:
                ax.set_xticks(range(len(labels)))
                ax.set_xticklabels(labels, rotation=45)
            else:
                ax.set_xticks([])
                ax.set_xticklabels([])

            # ---- Y AXIS: only first column ----
            if col == 0:
                ax.set_ylabel("Fraction")
            else:
                ax.set_yticks([])
                ax.set_ylabel("")

            ax.set_title(f'Head {head}')
            ax.set_xlabel('')
        else:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
            ax.set_title(f'Head {head}')
    fig.suptitle(f'Attention Fractions for Layer {selected_layer}')
    plt.tight_layout()
    plt.savefig(f'../results_llava-hf/llava-1.5-7b-hf/attention_fraction_analysis/layer{selected_layer}_fractions.png')
    plt.show()

This looks at the same as above, but for attention entropy and not attention fraction.

In [ ]:

# Specify the layers to plot
layers = range(0,32)  # Change this to the desired layers

selected_group_pairs = [("relation", "all_visual")]

for selected_layer in layers:
    # Collect entropies for each head in the selected layer per relation type
    entropy_per_head_relation = {}

    for result in results:
        relation_type = result["relation_type"]
        attn_metrics = result["attention_metrics"]
        for pair in attn_metrics["group_pairs"]:
            src = pair["source_group"]
            tgt = pair["target_group"]
            if (src, tgt) not in selected_group_pairs:
                continue
            per_layer = pair["per_layer"]
            layer_data = per_layer[selected_layer]
            per_head_entropy = layer_data["per_head_entropy"]
            for head_idx, entropy in enumerate(per_head_entropy):
                if head_idx not in entropy_per_head_relation:
                    entropy_per_head_relation[head_idx] = {}
                if relation_type not in entropy_per_head_relation[head_idx]:
                    entropy_per_head_relation[head_idx][relation_type] = []
                entropy_per_head_relation[head_idx][relation_type].append(entropy)

    # Now plot violin plots for each head in the selected layer, with violins for each relation type
    num_heads = 32
    fig, axes = plt.subplots(4, 8, figsize=(20, 10))  # 4 rows, 8 columns for 32 heads
    axes = axes.flatten()
    for head in range(num_heads):
        ax = axes[head]

        ax.set_ylim(2, 7)
        row = head // 8
        col = head % 8
        if head in entropy_per_head_relation:
            relation_entropy = entropy_per_head_relation[head]
            data = list(relation_entropy.values())
            labels = list(relation_entropy.keys())
            sns.violinplot(data=data, ax=ax, inner="quartile")

            # ---- X AXIS: only bottom row ----
            if row == 3:
                ax.set_xticks(range(len(labels)))
                ax.set_xticklabels(labels, rotation=45)
            else:
                ax.set_xticks([])
                ax.set_xticklabels([])

            # ---- Y AXIS: only first column ----
            if col == 0:
                ax.set_ylabel("Entropy")
            else:
                ax.set_yticks([])
                ax.set_ylabel("")

            ax.set_title(f'Head {head}')
            ax.set_xlabel('')
        else:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
            ax.set_title(f'Head {head}')
    fig.suptitle(f'Attention Entropy for Layer {selected_layer}')
    plt.tight_layout()
    plt.savefig(f'../results_llava-hf/llava-1.5-7b-hf/attention_entropy_analysis/layer{selected_layer}_entropy.png')
    plt.show()

# What layer's have significant attention to image tokens?

In [ ]:
# Compute the per-layer mean_fraction for (last, all_visual) averaged across all QAs
all_per_layer_fractions = []

for result in results:
    # Find the group pair with source_group == 'last' and target_group == 'all_visual'
    pair = next((p for p in result['attention_metrics']['group_pairs']
                 if p['source_group'] == 'last' and p['target_group'] == 'all_visual'), None)
    if pair is not None:
        # Collect the mean_fraction for each layer
        per_layer = pair['per_layer']
        fractions = [layer['mean_fraction'] for layer in per_layer]
        all_per_layer_fractions.append(fractions)

# Convert to numpy array for averaging (shape: num_qas, num_layers)
all_per_layer_fractions = np.array(all_per_layer_fractions)
mean_per_layer_fraction = np.mean(all_per_layer_fractions, axis=0)
print("Mean per-layer fraction (last → all_visual) across all QAs:")
print(mean_per_layer_fraction)

# Print layers with fraction above threshold
THRESHOLD = 0.2  # Set your threshold here
print(f"Layers with mean fraction > {THRESHOLD}:")
for i, frac in enumerate(mean_per_layer_fraction):
    if frac > THRESHOLD:
        print(f"Layer {i}: {frac:.3f}")

# Visualise the directionality of attention (CoM)

In [ ]:
import json

# Load all results from the new JSONL (line-based) file
results_com = []
with open("../results_llava-hf/llava-1.5-7b-hf/attention_results_with_com.jsonl", "r") as f:
    for line in f:
        if line.strip():
            results_com.append(json.loads(line))
print(f"Number of inferences: {len(results_com)}")

Attention Map CoM:  [11.54429622082459, 11.795020252410703]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.926155879987675, 13.40741261620169]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [9.759384164587185, 13.730961883599338]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [9.99943028599643, 14.085764237691262]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.02871902389638, 14.057345229654693]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [9.641011947855562, 13.221378962928707]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.208824118471743, 13.732386517595222]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.527037645044969, 14.758263421415334]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.25905019584633, 13.373999400789268]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.04696367406231, 14.977108798200089]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.003854961335858, 13.944532077525148]
Object CoM:  [5.0, 4.0]
Attention Map CoM:  [10.662341270327916, 14.043301759559471]
Object CoM:  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# CONFIG
# -------------------------
GRID_SIZE = 24
NUM_LAYERS = 32
NUM_HEADS = 32

SOURCE_GROUP = "relation"
TARGET_GROUP = "all_visual"

# Normalize distances by image diagonal
MAX_DIST = np.sqrt(2 * (GRID_SIZE - 1) ** 2)
DIST_THRESHOLD = 0.10  # 10% of image diagonal

# -------------------------
# COLLECT VECTORS
# -------------------------
# (relation_type, layer, head) -> list of displacement vectors
vectors = {}

for result in results:
    relation_type = result["relation_type"]
    attn_metrics = result["attention_metrics"]

    for pair in attn_metrics["group_pairs"]:
        if (
            pair["source_group"] != SOURCE_GROUP
            or pair["target_group"] != TARGET_GROUP
        ):
            continue

        for layer_idx, layer in enumerate(pair["per_layer"]):
            com_att_list = layer.get("CoM_Attention")  # list of 32 [x,y]
            com_obj = layer.get("CoM_Object")          # single [x,y]

            if com_att_list is None or com_obj is None:
                continue

            com_obj = np.asarray(com_obj)

            for head_idx, head_com in enumerate(com_att_list):
                head_com = np.asarray(head_com)
                delta = head_com - com_obj  # image-space displacement

                key = (relation_type, layer_idx, head_idx)
                vectors.setdefault(key, []).append(delta)

# -------------------------
# AVERAGE VECTORS
# -------------------------
avg_vectors = {}  # (relation, layer, head) -> mean vector

for key, vecs in vectors.items():
    avg_vectors[key] = np.mean(np.stack(vecs), axis=0)

print("Non-zero vectors per head:")

for h in range(NUM_HEADS):
    count = sum(
        (relation_type, l, h) in avg_vectors
        for l in range(NUM_LAYERS)
    )
    if count > 0:
        print(f"Head {h}: {count} layers")

# -------------------------
# VISUALIZE PER RELATION
# -------------------------
for relation_type in sorted(set(k[0] for k in avg_vectors.keys())):

    # Grid
    layers = np.arange(NUM_LAYERS)
    heads = np.arange(NUM_HEADS)
    X, Y = np.meshgrid(layers, heads)

    U = np.zeros((NUM_HEADS, NUM_LAYERS))
    V = np.zeros((NUM_HEADS, NUM_LAYERS))
    M = np.zeros((NUM_HEADS, NUM_LAYERS))  # normalized magnitude

    for l in range(NUM_LAYERS):
        for h in range(NUM_HEADS):
            key = (relation_type, l, h)
            if key not in avg_vectors:
                continue

            vec = avg_vectors[key]

            # Image-space → plot-space
            U[h, l] = vec[0]
            V[h, l] = -vec[1]  # flip Y for image coordinates

            M[h, l] = np.linalg.norm(vec) / MAX_DIST

    # Mask weak vectors
    mask = M < DIST_THRESHOLD
    U = np.ma.masked_where(mask, U)
    V = np.ma.masked_where(mask, V)

    # -------------------------
    # PLOT
    # -------------------------
    plt.figure(figsize=(16, 10))

    Q = plt.quiver(
        X, Y, U, V, M,
        angles="xy",
        scale_units="xy",
        scale=1,
        cmap="plasma",
        pivot="mid",
        width=0.003
    )

    cbar = plt.colorbar(Q)
    cbar.set_label("Normalized CoM Distance")

    plt.xlabel("Layer index (grid position)")
    plt.ylabel("Head index (grid position)")
    plt.title(f"Attention Shift Map — Relation: {relation_type}")

    plt.xticks(np.arange(0, NUM_LAYERS, 2))
    plt.yticks(np.arange(0, NUM_HEADS, 2))
    plt.grid(alpha=0.15)

    plt.tight_layout()
    plt.show()


TypeError: object of type 'NoneType' has no len()